# Qwen3-4B Sinhala QA Fine-Tuning

LoRA fine-tune Qwen3-4B for grounded Sinhala QA, using the same pipeline architecture as
`llama-scripts/qa-finetuning_v6.ipynb`. Two modes, controlled by `USE_CPT_CHECKPOINT` in
the config cell:

- **`True` (default): fine-tune on top of `isji/qwen3-4b-cpt`** — the Sinhala continual
  pre-training checkpoint (`qwen3-4b-sinhala-cpt-b200.ipynb`), which used the
  `isji/Extended-Sinhala-Qwen3` tokenizer (176,856 tokens, +25,187 Sinhala). This is the
  full three-stage pipeline: tokenizer extension → CPT → QA fine-tuning.
- **`False`: fine-tune stock `Qwen/Qwen3-4B`** (original vocabulary, no CPT) — kept as a
  toggle rather than removed, so the two can be run as a controlled ablation (does CPT
  actually help once QA fine-tuning is applied to both?), not just compared to the
  zero-shot baseline.

Kept from v6: canonical refusal targets, answer-aware evidence windows, context-group
validation split, unanswerable rebalancing, prompt/completion records with completion-only
loss, deterministic inference with top-window retrieval and a lexical grounding gate, and
the same external-test metrics (normalized exact match, token F1, false-answer rate).

Adapted for Qwen3-4B:

- **Chat-model training, not raw completion.** v6 fed a base Llama raw
  instruction+context+answer text. Qwen3-4B is instruction-tuned, so every example is
  rendered through the **Qwen chat template** with the *same English system prompt the
  Qwen evaluation harness uses* (`qwen3-4b-instruct-2507-test-split-inference.ipynb`) —
  train/eval prompt distributions match exactly. Targets stay string prompt/completion
  pairs, so TRL's `completion_only_loss` applies loss only to the answer.
- **Thinking mode disabled.** `Qwen/Qwen3-4B` is a hybrid thinking model; prompts are
  rendered with `enable_thinking=False` (auto-detected from the chat template — verified
  present on both `Qwen/Qwen3-4B` and the CPT tokenizer, which inherited it from
  `Qwen/Qwen3-4B-Base`).
- **Tokenizer-dependent token budgets.** The default Qwen tokenizer manages 9.19
  tokens/word on this data (byte-level fallback) and gold answers reach 261 tokens, so
  `MAX_NEW_TOKENS=320` (the 48 used by v6 would truncate most answers). The CPT tokenizer
  is far more efficient (1.48 tokens/word) but the budget is kept the same for both modes
  so the two are trained/evaluated identically. `MAX_LENGTH=2048` fits the longest
  system+context+question+answer in either case (audited below).
- **Special-token handling.** Rendered chat strings already contain `<|im_start|>` etc.,
  so all tokenization of rendered prompts uses `add_special_tokens=False`. Qwen has no BOS
  token; pad (`<|endoftext|>`) and eos (`<|im_end|>`) are distinct out of the box.
- **4B-scale hyperparameters** for a single B200: micro-batch 4 × grad-accum 8 (the same
  32-example effective batch as v6), LoRA r=32/α=64 on all projections, lr 1e-4, bf16.

## Loading the CPT checkpoint correctly (read this before changing the model cell)

`isji/qwen3-4b-cpt` is a **LoRA adapter**, not a merged model — it must be loaded onto
`Qwen/Qwen3-4B` and merged, not passed directly as `MODEL_ID`. Two details from the CPT
notebook carry over and are easy to silently get wrong:

1. **Embeddings must be resized before the adapter loads**, since the adapter's saved
   `embed_tokens` tensor is already the extended (176,856, hidden) shape and will
   shape-mismatch against the stock model's (151,669, hidden) embedding table otherwise.
2. **The tied-embedding fix must be reapplied here, not just trusted from the CPT run.**
   Qwen3-4B ties `embed_tokens`/`lm_head` into one matrix. Loading a fresh base model
   already re-ties them via `tie_weights()` — but wrapping `embed_tokens` in PEFT's
   `ModulesToSaveWrapper` (which `PeftModel.from_pretrained` does to reconstruct
   `modules_to_save=["embed_tokens"]`) creates the trainable copy via `deepcopy`, which
   is a **new, separate tensor** — `lm_head` stays aliased to the original, frozen
   embedding, not the one about to be overwritten with the adapter's trained weights.
   This is the exact same bug fixed during CPT, but it re-occurs on every fresh load
   because the alias is a Python object relationship, not something serialized in the
   adapter files. The loading cell below re-ties `lm_head` immediately after
   `PeftModel.from_pretrained` (before merging) and verifies the merged result by value,
   not by assumption.

The merged model is what `qwen3-4b-instruct-2507-test-split-inference.ipynb` consumes for
the final comparison; this notebook also runs the v6-style external evaluation itself.

In [ ]:
%uv pip install -q "transformers>=4.51,<5" "trl>=0.26,<0.30" "peft>=0.19" datasets accelerate huggingface_hub hf_transfer tqdm

In [ ]:
import hashlib
import json
import math
import os
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import set_seed

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

SEED = 42

# ---- Model selection ----
# True  : fine-tune isji/qwen3-4b-cpt (Sinhala CPT + extended tokenizer) — the full pipeline.
# False : fine-tune stock BASE_MODEL_ID directly — kept as a toggle for a CPT-vs-no-CPT
#         ablation under identical QA fine-tuning, not just a comparison to zero-shot.
USE_CPT_CHECKPOINT = True
BASE_MODEL_ID = "Qwen/Qwen3-4B"  # swap to "Qwen/Qwen3-4B-Instruct-2507" for the non-hybrid variant
CPT_ADAPTER_ID = "isji/qwen3-4b-cpt"

MODEL_ID = CPT_ADAPTER_ID if USE_CPT_CHECKPOINT else BASE_MODEL_ID  # for logging/repo naming only

TRAIN_CANDIDATES = [
    Path("/tmp/train.jsonl"),
    Path("new_split_v2/train.jsonl"),
    Path("../new_split_v2/train.jsonl"),
]
TRAIN_PATH = next((p for p in TRAIN_CANDIDATES if p.is_file()), TRAIN_CANDIDATES[0])
TEST_CANDIDATES = [
    Path("/tmp/test_updated.jsonl"),
    Path("/tmp/test.jsonl"),
    Path("new_split_v2/test.jsonl"),
    Path("../new_split_v2/test.jsonl"),
]
TEST_PATH = next((p for p in TEST_CANDIDATES if p.is_file()), TEST_CANDIDATES[0])

# Variant-tagged paths/repos so a CPT run and a no-CPT run never silently overwrite each
# other — both are needed side by side for the ablation.
_VARIANT = "cpt" if USE_CPT_CHECKPOINT else "nocpt"
OUTPUT_DIR = Path(f"/tmp/output_qwen3_4b_qa_{_VARIANT}")
QA_ADAPTER_DIR = Path(f"/tmp/qwen3_4b_qa_{_VARIANT}_adapter")
MERGED_MODEL_DIR = Path(f"/tmp/qwen3_4b_qa_{_VARIANT}_merged")
RESULTS_PATH = Path(f"/tmp/qwen3_4b_qa_{_VARIANT}_results.jsonl")
MERGED_REPO_ID = os.environ.get("QA_MERGED_REPO_ID", f"isji/qwen3-4b-sinhala-qa-{_VARIANT}-merged")
ADAPTER_REPO_ID = os.environ.get("QA_ADAPTER_REPO_ID", f"isji/qwen3-4b-sinhala-qa-{_VARIANT}-adapter")
HF_REPO_PRIVATE = True

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 320          # default Qwen tokenizer: gold answers reach 261 tokens
VALIDATION_FRACTION = 0.02
MIN_UNANSWERABLE_TRAIN_FRACTION = 0.25
TOP_K_WINDOWS = 2
GROUNDING_THRESHOLD = 0.50

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("Mode          :", "CPT checkpoint + QA fine-tune" if USE_CPT_CHECKPOINT else "stock model + QA fine-tune (no CPT)")
print("Base model    :", BASE_MODEL_ID)
if USE_CPT_CHECKPOINT:
    print("CPT adapter   :", CPT_ADAPTER_ID)
print("Training data :", TRAIN_PATH, "| exists:", TRAIN_PATH.is_file())
print("External test :", TEST_PATH, "| exists:", TEST_PATH.is_file())

In [ ]:
# Only needed for private data/repos or pushing. Requires HF_TOKEN in the environment.
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])

In [ ]:
from transformers import AutoTokenizer

# The CPT adapter's tokenizer is the extended one (176,856 tokens); the stock path uses
# whatever BASE_MODEL_ID ships with. Either way, IDs must match what the loaded model uses.
tokenizer_source = CPT_ADAPTER_ID if USE_CPT_CHECKPOINT else BASE_MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"

# Qwen/Qwen3-4B is a hybrid thinking model whose template accepts enable_thinking; the CPT
# tokenizer inherited the same template from Qwen3-4B-Base (verified present).
# Qwen3-4B-Instruct-2507's template has no thinking mode and ignores the flag.
TEMPLATE_SUPPORTS_THINKING = "enable_thinking" in (tokenizer.chat_template or "")
CHAT_KWARGS = {"enable_thinking": False} if TEMPLATE_SUPPORTS_THINKING else {}

print("Tokenizer source     :", tokenizer_source)
print("Tokenizer size       :", len(tokenizer))
print("eos / pad            :", tokenizer.eos_token, "/", tokenizer.pad_token)
print("Thinking-mode switch :", "present -> disabled" if TEMPLATE_SUPPORTS_THINKING else "not in template (already non-thinking)")

In [ ]:
from transformers import AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this notebook.")
train_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading base model {BASE_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=train_dtype,
    device_map={"": torch.cuda.current_device()},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)

if USE_CPT_CHECKPOINT:
    from peft import PeftModel

    original_vocab_size = model.get_input_embeddings().num_embeddings
    if original_vocab_size != len(tokenizer):
        model.resize_token_embeddings(len(tokenizer), mean_resizing=True)
        print(f"Resized embeddings: {original_vocab_size:,} -> {len(tokenizer):,}")

    input_weight = model.get_input_embeddings().weight
    output_weight = model.get_output_embeddings().weight
    embeddings_are_tied = input_weight.data_ptr() == output_weight.data_ptr()
    print(f"Embeddings tied after resize: {embeddings_are_tied}")

    print(f"Loading and merging CPT adapter {CPT_ADAPTER_ID}...")
    model = PeftModel.from_pretrained(model, CPT_ADAPTER_ID)

    # ---- Re-apply the tied-embedding fix (see the intro markdown for why this is
    # necessary on every fresh load, not just during the original CPT training run) ----
    if embeddings_are_tied:
        base_model = model.base_model.model
        embed_wrapper = base_model.model.embed_tokens
        if hasattr(embed_wrapper, "modules_to_save"):
            trainable_embedding = embed_wrapper.modules_to_save["default"].weight
            head_untied = base_model.lm_head.weight.data_ptr() != trainable_embedding.data_ptr()
            if head_untied:
                base_model.lm_head.weight = embed_wrapper.modules_to_save["default"].weight
                print("Re-tied lm_head to the loaded, trained embedding copy.")
            if base_model.lm_head.weight.data_ptr() != trainable_embedding.data_ptr():
                raise RuntimeError(
                    "lm_head is not tied to the trained embedding copy after reloading the "
                    "CPT adapter — the new Sinhala tokens would be unusable for generation. "
                    "Do not proceed to merge or train in this state."
                )
            print("Tie verified before merge: lm_head shares the trained embedding matrix.")

    pre_merge_embedding = model.base_model.model.model.embed_tokens.modules_to_save[
        "default"
    ].weight.detach().clone()

    print("Merging CPT adapter into the base model...")
    model = model.merge_and_unload()

    # ---- Verify the merge by value, not by assumption ----
    merged_input = model.get_input_embeddings().weight
    merged_output = model.get_output_embeddings().weight
    if merged_input.shape[0] != len(tokenizer) or merged_output.shape[0] != len(tokenizer):
        raise RuntimeError(
            f"Merged model vocabulary ({merged_input.shape[0]:,}) does not match the "
            f"tokenizer ({len(tokenizer):,})."
        )
    if not torch.equal(merged_input, merged_output):
        raise RuntimeError(
            "Merged input/output embeddings differ — lm_head did not receive the trained "
            "CPT embedding values. This is the tied-embedding bug resurfacing; do not train "
            "or evaluate on this model."
        )
    if not torch.equal(merged_input.detach().cpu(), pre_merge_embedding.cpu()):
        raise RuntimeError(
            "Merged embedding values differ from the loaded adapter's trained embedding — "
            "merge_and_unload() did not apply the expected values."
        )
    print(f"Merge verified: input/output embeddings match, both ({merged_input.shape[0]:,}, "
          f"{merged_input.shape[1]:,}), and equal the loaded CPT-trained values.")

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
print("Model ready:", MODEL_ID, "| dtype:", train_dtype)

In [ ]:
# Same system prompt as the Qwen evaluation harness, so training and evaluation share one
# prompt distribution. The canonical refusal string matches the gold labels in the dataset.
SYSTEM_PROMPT = f"""You are a helpful Sinhala history question-answering assistant.

Your task is to answer the question using ONLY the information explicitly provided in the context.

Instructions:

- Read the entire context carefully before answering.
- Use only the information explicitly stated in the context.
- Do not use external knowledge, assumptions, or prior knowledge.
- Identify the exact information requested by the question.
- If the answer is found in multiple parts of the context, combine the relevant information into a single complete answer.
- Include only information that directly answers the question.
- Do not include additional facts, names, dates, or events unless they are required to answer the question.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and the requested attribute.
- Do not take a date or fact from a neighboring sentence about a different entity or event.
- Do not infer or guess information that is not explicitly stated, except for simple arithmetic explicitly requested by the question when all required values are stated in the context.
- For a duration question with explicit starting and ending years, subtract the starting year from the ending year and return the duration.
- If the answer cannot be found in the context, respond exactly with:
  "{NO_ANSWER}"
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning.
- Do not mention passage numbers, page numbers, chapter names, grades, or any other source references.
- Answer in a single line. Do not add a preamble, a label, or quotation marks."""

SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def build_messages(context, question):
    user_prompt = f"""Context:

{clean_text(context)}

Question:

{clean_text(question)}

Answer:"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def render_prompt(context, question):
    """Chat-template-rendered prompt string ending at the assistant generation position."""
    return tokenizer.apply_chat_template(
        build_messages(context, question),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_KWARGS,
    )


def prompt_token_count(text):
    # Rendered strings already carry <|im_start|> etc. as text, so no extra specials.
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSONL file not found: {path}")

    records = []
    fingerprints = set()
    dropped = 0
    duplicates = 0

    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error

            question = clean_text(item.get("question"))
            context = clean_text(item.get("context"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))

            normalized = {
                "question": question,
                "context": context,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
            }
            if not question or not context or (answerable and not normalized["answer"]):
                dropped += 1
                continue

            fingerprint = (
                normalized["question"],
                normalized["context"],
                normalized["answer"],
                normalized["answerable"],
            )
            if fingerprint in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fingerprint)
            records.append(normalized)

    print(f"Loaded {len(records):,} unique records from {path}")
    print(f"Dropped invalid/empty: {dropped:,}; exact duplicates removed: {duplicates:,}")
    return records


records = load_jsonl(TRAIN_PATH)
answerable_count = sum(item["answerable"] for item in records)
print(f"Answerable: {answerable_count:,}; unanswerable: {len(records) - answerable_count:,}")

print("\n--- Rendered prompt preview (tail) ---")
print(render_prompt(records[0]["context"], records[0]["question"])[-600:])

In [ ]:
def token_supported(token, normalized_context):
    if token in normalized_context:
        return True
    # Sinhala case endings often add one character; a short stem check preserves grounded variants.
    return len(token) >= 4 and token[:-1] in normalized_context


def rank_context_windows(context, query, token_budget, top_k=1):
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    if len(context_ids) <= token_budget:
        return [(context, 1.0)]

    query_tokens = set(lexical_tokens(query))
    stride = max(64, token_budget // 2)
    candidates = []

    for start in range(0, len(context_ids), stride):
        chunk_ids = context_ids[start : start + token_budget]
        if len(chunk_ids) < 32:
            continue
        chunk = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()
        normalized_chunk = " ".join(lexical_tokens(chunk))
        if query_tokens:
            matched = sum(token_supported(token, normalized_chunk) for token in query_tokens)
            score = matched / len(query_tokens)
        else:
            score = 0.0
        candidates.append((chunk, score, start))
        if start + token_budget >= len(context_ids):
            break

    candidates.sort(key=lambda value: (-value[1], value[2]))
    return [(chunk, score) for chunk, score, _ in candidates[:top_k]]


def context_budget(question, completion):
    fixed_tokens = prompt_token_count(render_prompt("", question))
    completion_tokens = len(tokenizer(completion, add_special_tokens=False)["input_ids"])
    return max(128, MAX_LENGTH - fixed_tokens - completion_tokens - 24)


def make_training_example(item):
    answer = canonical_answer(item)
    completion = answer + tokenizer.eos_token
    budget = context_budget(item["question"], completion)
    query = item["question"] if not item["answerable"] else f"{item['question']} {answer}"
    window = rank_context_windows(item["context"], query, budget, top_k=1)[0][0]
    prompt = render_prompt(window, item["question"])

    # Keep a safety loop so TRL's right truncation can never remove completion supervision.
    for _ in range(2):
        total_length = prompt_token_count(prompt + completion)
        if total_length <= MAX_LENGTH - 4:
            break
        budget = max(128, budget - (total_length - MAX_LENGTH) - 16)
        window = rank_context_windows(item["context"], query, budget, top_k=1)[0][0]
        prompt = render_prompt(window, item["question"])

    return {"prompt": prompt, "completion": completion}


# Split on context hashes, not individual rows, to prevent the same passage leaking into validation.
groups = {}
for item in records:
    key = hashlib.sha1(item["context"].encode("utf-8")).hexdigest()
    groups.setdefault(key, []).append(item)

group_keys = sorted(groups)
random.Random(SEED).shuffle(group_keys)
# Use at least 200 validation contexts for small datasets, capped at 20%.
desired_validation_groups = max(200, round(len(group_keys) * VALIDATION_FRACTION))
validation_group_cap = max(1, math.floor(len(group_keys) * 0.20))
validation_group_count = min(
    desired_validation_groups,
    validation_group_cap,
    max(1, len(group_keys) - 1),
)
validation_keys = set(group_keys[:validation_group_count])

train_records = []
validation_records = []
for key, group in groups.items():
    (validation_records if key in validation_keys else train_records).extend(group)


def rebalance_unanswerable(items, minimum_fraction):
    positives = [item for item in items if item["answerable"]]
    negatives = [item for item in items if not item["answerable"]]
    if not negatives or len(negatives) / len(items) >= minimum_fraction:
        return list(items), 0

    required_negative_count = math.ceil(
        minimum_fraction * len(positives) / (1.0 - minimum_fraction)
    )
    extra_count = max(0, required_negative_count - len(negatives))
    rng = random.Random(SEED)
    balanced = list(items) + [rng.choice(negatives) for _ in range(extra_count)]
    rng.shuffle(balanced)
    return balanced, extra_count


train_records, repeated_negative_count = rebalance_unanswerable(
    train_records,
    MIN_UNANSWERABLE_TRAIN_FRACTION,
)

print(f"Train records after balancing: {len(train_records):,}")
print(f"Repeated hard-negative rows added: {repeated_negative_count:,}")
print(f"Validation records: {len(validation_records):,}")
print("Preparing answer-preserving prompt/completion examples...")

train_examples = [make_training_example(item) for item in tqdm(train_records, desc="Train")]
validation_examples = [make_training_example(item) for item in tqdm(validation_records, desc="Validation")]

from datasets import Dataset

train_dataset = Dataset.from_list(train_examples)
validation_dataset = Dataset.from_list(validation_examples)

# Audit a sample after preprocessing.
audit_sample = random.Random(SEED).sample(
    train_examples,
    min(1000, len(train_examples)),
)
audit_lengths = [prompt_token_count(item["prompt"] + item["completion"]) for item in audit_sample]
print(f"Audited maximum sequence length: {max(audit_lengths):,}/{MAX_LENGTH:,}")
print("\n--- Prepared example (prompt tail + completion) ---")
print(train_dataset[0]["prompt"][-400:])
print(train_dataset[0]["completion"])

In [ ]:
from peft import LoraConfig, TaskType
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

qa_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

training_epochs = 2 if len(train_dataset) >= 50_000 else 3
print(f"Training epochs selected for dataset size: {training_epochs}")
updates_per_epoch = math.ceil(
    len(train_dataset) / (4 * 8)
)
estimated_total_updates = max(1, math.ceil(updates_per_epoch * training_epochs))
evaluation_steps = max(10, min(100, estimated_total_updates // 10))
print(f"Estimated optimizer updates: {estimated_total_updates:,}")
print(f"Evaluation/save interval: every {evaluation_steps:,} updates")

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    eval_packing=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=training_epochs,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=(train_dtype == torch.bfloat16),
    fp16=(train_dtype == torch.float16),
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=evaluation_steps,
    save_strategy="steps",
    save_steps=evaluation_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    logging_first_step=True,
    group_by_length=True,
    dataset_num_proc=min(8, os.cpu_count() or 1),
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=qa_lora_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.model.print_trainable_parameters()
print("Starting Sinhala QA fine-tuning on stock Qwen3-4B...")
train_result = trainer.train()
print(train_result)

In [ ]:
# Save the lightweight QA adapter first.
QA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(QA_ADAPTER_DIR))
tokenizer.save_pretrained(QA_ADAPTER_DIR)

# Also save one deployable merged model for the shared evaluation harness.
print("Merging the trained QA adapter into the base model...")
model = trainer.model.merge_and_unload()
model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
# Deterministic evaluation: clear Qwen's shipped sampling preset on the merged model.
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
model.eval()

MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(
    MERGED_MODEL_DIR,
    safe_serialization=True,
    max_shard_size="4GB",
)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("QA adapter:", QA_ADAPTER_DIR)
print("Deployable merged model:", MERGED_MODEL_DIR)

In [ ]:
def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


def evidence_support(answer, context):
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))
    supported = sum(token_supported(token, normalized_context) for token in answer_tokens)
    return supported / len(answer_tokens)


def generate_candidate(context_window, question):
    prompt = render_prompt(context_window, question)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=model.generation_config.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1] :]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.splitlines()[0].strip(" []{}()<>\"'`") if answer else ""
    return answer


def run_qa(context, question, use_grounding=True):
    completion_stub = NO_ANSWER + tokenizer.eos_token
    budget = context_budget(question, completion_stub)
    windows = rank_context_windows(context, question, budget, top_k=TOP_K_WINDOWS)
    candidates = []

    for window, retrieval_score in windows:
        raw_answer = generate_candidate(window, question)
        support = 1.0 if is_no_answer(raw_answer) else evidence_support(raw_answer, window)
        candidates.append({
            "raw_answer": raw_answer,
            "window": window,
            "retrieval_score": retrieval_score,
            "support": support,
        })

    grounded = [
        candidate for candidate in candidates
        if candidate["raw_answer"] and not is_no_answer(candidate["raw_answer"])
        and candidate["support"] >= GROUNDING_THRESHOLD
    ]

    if grounded:
        best = max(
            grounded,
            key=lambda candidate: (candidate["support"], candidate["retrieval_score"]),
        )
        final_answer = best["raw_answer"]
    else:
        best = max(candidates, key=lambda candidate: candidate["retrieval_score"])
        final_answer = NO_ANSWER if use_grounding else best["raw_answer"]

    return {
        "answer": final_answer,
        "raw_answer": best["raw_answer"],
        "support": best["support"],
        "retrieval_score": best["retrieval_score"],
        "candidate_count": len(candidates),
    }


print("Grounded run_qa(context, question) is ready.")

In [ ]:
def token_f1(prediction, reference):
    prediction_digits = re.findall(r"\d+", normalize_answer(prediction))
    reference_digits = re.findall(r"\d+", normalize_answer(reference))
    if reference_digits and prediction_digits != reference_digits:
        return 0.0
    prediction_tokens = lexical_tokens(prediction)
    reference_tokens = lexical_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0
    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


if not TEST_PATH.is_file():
    print(f"Skipping external evaluation because {TEST_PATH} is not present.")
else:
    test_records = load_jsonl(TEST_PATH)
    exact_correct = 0
    raw_exact_correct = 0
    f1_total = 0.0
    answerable_correct = 0
    answerable_total = 0
    unanswerable_correct = 0
    unanswerable_total = 0
    unsupported_rejections = 0
    predicted_no_answer_count = 0
    correct_no_answer_count = 0

    with RESULTS_PATH.open("w", encoding="utf-8", newline="\n") as results_file:
        for index, item in enumerate(test_records, 1):
            reference = canonical_answer(item)
            result = run_qa(item["context"], item["question"], use_grounding=True)
            prediction = result["answer"]
            raw_prediction = result["raw_answer"]

            exact = normalize_answer(prediction) == normalize_answer(reference)
            raw_exact = normalize_answer(raw_prediction) == normalize_answer(reference)
            f1 = token_f1(prediction, reference)
            predicted_no_answer = is_no_answer(prediction)
            exact_correct += int(exact)
            raw_exact_correct += int(raw_exact)
            f1_total += f1
            predicted_no_answer_count += int(predicted_no_answer)
            correct_no_answer_count += int(predicted_no_answer and not item["answerable"])
            unsupported_rejections += int(
                prediction == NO_ANSWER and raw_prediction and not is_no_answer(raw_prediction)
            )

            if item["answerable"]:
                answerable_total += 1
                answerable_correct += int(exact)
            else:
                unanswerable_total += 1
                unanswerable_correct += int(exact)

            saved = {
                "index": index,
                "question": item["question"],
                "reference": reference,
                "raw_prediction": raw_prediction,
                "prediction": prediction,
                "answerable": item["answerable"],
                "exact_match": exact,
                "token_f1": f1,
                "evidence_support": result["support"],
                "retrieval_score": result["retrieval_score"],
            }
            results_file.write(json.dumps(saved, ensure_ascii=False) + "\n")
            results_file.flush()

            print("\n" + "=" * 100, flush=True)
            print(f"[{index}/{len(test_records)}]", flush=True)
            print("Question :", item["question"], flush=True)
            print("Reference:", reference, flush=True)
            print("Raw      :", raw_prediction, flush=True)
            print("Final    :", prediction, flush=True)
            print(f"Support  : {result['support']:.3f}", flush=True)
            print(f"Exact/F1 : {exact} / {f1:.3f}", flush=True)

    total = len(test_records)
    print("\n" + "=" * 100)
    print("EXTERNAL TEST RESULTS")
    print("=" * 100)
    print(f"Grounded exact match : {exact_correct}/{total} ({100 * exact_correct / total:.2f}%)")
    print(f"Raw exact match      : {raw_exact_correct}/{total} ({100 * raw_exact_correct / total:.2f}%)")
    print(f"Mean token F1        : {f1_total / total:.4f}")
    print(f"Answerable exact     : {answerable_correct}/{answerable_total}")
    print(f"Unanswerable exact   : {unanswerable_correct}/{unanswerable_total}")
    false_answers = unanswerable_total - unanswerable_correct
    no_answer_precision = correct_no_answer_count / max(predicted_no_answer_count, 1)
    no_answer_recall = correct_no_answer_count / max(unanswerable_total, 1)
    no_answer_f1 = (
        2 * no_answer_precision * no_answer_recall / (no_answer_precision + no_answer_recall)
        if no_answer_precision + no_answer_recall else 0.0
    )
    print(f"False-answer rate on unanswerable: {false_answers}/{unanswerable_total} "
          f"({100 * false_answers / max(unanswerable_total, 1):.2f}%)")
    print(f"No-answer precision/recall/F1: {no_answer_precision:.4f} / "
          f"{no_answer_recall:.4f} / {no_answer_f1:.4f}")
    print(f"Unsupported generations rejected: {unsupported_rejections}")
    print("Detailed results:", RESULTS_PATH)

## Push the adapter and the merged model to Hugging Face

The merged model is what the shared evaluation harness
(`qwen3-4b-instruct-2507-test-split-inference.ipynb`) loads as `MODEL_ID`; the adapter is
the lightweight artifact (attachable to stock `Qwen/Qwen3-4B` with PEFT). Set the optional
environment variables `QA_MERGED_REPO_ID` / `QA_ADAPTER_REPO_ID` to override the default
repository names. Requires `HF_TOKEN` in the environment.

In [ ]:
from huggingface_hub import HfApi

print(f"Pushing QA adapter to: {ADAPTER_REPO_ID}")
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(ADAPTER_REPO_ID, private=HF_REPO_PRIVATE, exist_ok=True)
api.upload_folder(
    repo_id=ADAPTER_REPO_ID,
    folder_path=str(QA_ADAPTER_DIR),
    commit_message="Upload Qwen3-4B Sinhala QA POC LoRA adapter",
)
print(f"Adapter pushed to https://huggingface.co/{ADAPTER_REPO_ID}")

print(f"\nPushing merged QA model to: {MERGED_REPO_ID}")
model.push_to_hub(
    MERGED_REPO_ID,
    token=os.environ["HF_TOKEN"],
    safe_serialization=True,
    max_shard_size="5GB",
    private=HF_REPO_PRIVATE,
    commit_message="Upload Qwen3-4B Sinhala QA POC merged model",
)
tokenizer.push_to_hub(
    MERGED_REPO_ID,
    token=os.environ["HF_TOKEN"],
    private=HF_REPO_PRIVATE,
    commit_message="Upload matching tokenizer",
)
print(f"Merged model pushed to https://huggingface.co/{MERGED_REPO_ID}")

## After this POC

1. **Compare against zero-shot.** Point `MODEL_ID` in
   `qwen3-4b-instruct-2507-test-split-inference.ipynb` at the merged repo and rerun — the
   per-item report format is identical to `qwen/qwen-4B-instruct.txt` and
   `qwen/3B-v6-results.txt`, so the three systems diff directly.
2. **If evaluating the fine-tuned hybrid `Qwen/Qwen3-4B` in that harness**, make sure its
   `apply_chat_template` calls also pass `enable_thinking=False` — this notebook trained
   with thinking disabled, and evaluating with it enabled would mismatch the trained
   format. (Irrelevant if `MODEL_ID` was the Instruct-2507 variant.)
3. **The CPT arm comes next**: CPT `Qwen/Qwen3-4B` with `isji/Extended-Sinhala-Qwen3`,
   then rerun THIS notebook against the CPT checkpoint (swap `MODEL_ID`, and load the
   extended tokenizer) for the full pipeline comparison.

## Full test-split inference transcript

Standalone inference pass over the raw `test.jsonl` — keeping the grade/chapter metadata
that `load_jsonl` normalizes away — printed in the exact per-item block format of
`qwen/qwen-4B-instruct.txt` and `qwen/3B-v6-results.txt`, and saved to a text file so the
three systems can be diffed directly. Requires the merged model from the cells above to be
in memory (rerun the merge cell first if the kernel restarted).

In [ ]:
TRANSCRIPT_PATH = Path("/tmp/qwen3_4b_qa_poc_transcript.txt")

raw_test_rows = []
with TEST_PATH.open("r", encoding="utf-8-sig") as handle:
    for line in handle:
        if line.strip():
            raw_test_rows.append(json.loads(line))
print(f"Loaded {len(raw_test_rows)} raw test rows from {TEST_PATH}")

transcript_lines = []
exact_total = 0
f1_sum = 0.0

for index, row in enumerate(raw_test_rows, 1):
    reference = canonical_answer(row)
    result = run_qa(row["context"], row["question"], use_grounding=True)
    prediction = result["answer"]
    exact = normalize_answer(prediction) == normalize_answer(reference)
    f1 = token_f1(prediction, reference)
    exact_total += int(exact)
    f1_sum += f1

    block = [
        "=" * 100,
        f"[{index}/{len(raw_test_rows)}]",
        f"Grade    : {row.get('grade')} | Chapter: {row.get('chapter')}",
        f"Answerable: {row.get('answerable')}",
        f"Question : {row['question']}",
        f"Reference: {reference}",
        f"Raw      : {result['raw_answer']}",
        f"Final    : {prediction}",
        f"Support  : {result['support']:.3f}",
        f"Exact/F1 : {exact} / {f1:.3f}",
    ]
    print("\n".join(block), flush=True)
    transcript_lines.extend(block)

summary = [
    "=" * 100,
    f"TRANSCRIPT SUMMARY — {MODEL_ID} + QA POC fine-tune (merged)",
    f"Exact match  : {exact_total}/{len(raw_test_rows)} "
    f"({100 * exact_total / max(len(raw_test_rows), 1):.2f}%)",
    f"Mean token F1: {f1_sum / max(len(raw_test_rows), 1):.4f}",
    "=" * 100,
]
print("\n".join(summary))
transcript_lines.extend(summary)

with TRANSCRIPT_PATH.open("w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(transcript_lines) + "\n")
print("Saved transcript to", TRANSCRIPT_PATH)